# SG-DMNMF — Stability-Guided Deep Motif NMF for Community Detection

CSET347 minor project · Arpit Sahu (S24BCAU0195) · Team 8.

This notebook is **self-contained**: it clones the project from GitHub, installs the
dependencies, runs the Karate sanity check, reproduces the quick experiment, and displays
the generated tables and figures. It does not modify the research code or the results.

In [ ]:
# 1. Get the project (works in Colab and locally)
import os, sys, subprocess, tempfile

REPO = "https://github.com/ARPIT-KUMAR-SAHU1/motif-weight-deep-nmf.git"
BRANCH = "feature/complete-project"
BASE = "/content" if os.path.isdir("/content") else tempfile.gettempdir()
PROJECT = os.path.join(BASE, "sg-dmnmf")

if not os.path.isdir(os.path.join(PROJECT, ".git")):
    subprocess.check_call(["git", "clone", "-q", "-b", BRANCH, REPO, PROJECT])

os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
print("Project:", PROJECT)
print("Files:", sorted(os.listdir(PROJECT))[:8])

In [ ]:
# 2. Install dependencies
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", os.path.join(PROJECT, "requirements.txt")])
print("Dependencies installed.")

In [ ]:
# 3. Sanity check: the deep model must match the classic baseline on Karate
from src.data import load_karate
from src.models import DMRALite, SymNMF
from src.evaluate import evaluate_all
from src.motif import count_triangles

A, y, name = load_karate()
k = len(set(y))
print(f"{name}: n={A.shape[0]}, edges={int(A.sum())//2}, k={k}, triangles={count_triangles(A)}")

sym = evaluate_all(y, SymNMF(k=k, seed=42).fit(A))
deep = evaluate_all(y, DMRALite(k=k, alpha=0.5, max_iter=150, seed=42).fit(A))
print(f"SymNMF     ACC={sym['ACC']:.3f}  NMI={sym['NMI']:.3f}")
print(f"DMRA-lite  ACC={deep['ACC']:.3f}  NMI={deep['NMI']:.3f}")

## Reproduce the experiments

The next cell runs the same command as the local project:
`python experiments/run_all.py --quick`. It writes `results/` and `figures/`.

In [ ]:
# 4. Run the quick experiment pipeline
res = subprocess.run([sys.executable, "experiments/run_all.py", "--quick"],
                     cwd=PROJECT, capture_output=True, text=True)
print("exit code:", res.returncode)
print(res.stdout[-2000:])
if res.returncode != 0:
    print(res.stderr[-2000:])

In [ ]:
# 5. Display the generated E1 benchmark table
import pandas as pd
e1 = pd.read_csv(os.path.join(PROJECT, "results", "e1_clean_benchmark.csv"))
cols = ["Dataset", "Method", "ACC_mean", "NMI_mean", "ARI_mean", "AMI_mean", "Modularity_mean"]
display(e1[cols].round(3))

In [ ]:
# 6. Display alpha sensitivity and stability values
e2 = pd.read_csv(os.path.join(PROJECT, "results", "e2_alpha_sensitivity.csv"))
metric = e2["Metric"].iloc[0]
display(e2[e2["Metric"] == metric][["Dataset", "Alpha", "Mean", "Stability"]].round(3))

In [ ]:
# 7. Show the generated matplotlib figures
from IPython.display import Image, display
import glob
for pattern in ["e2_alpha_sensitivity_*.png", "e3_robustness_*_edge_deletion.png"]:
    for f in sorted(glob.glob(os.path.join(PROJECT, "figures", pattern))):
        print(os.path.basename(f))
        display(Image(filename=f, width=700))

## How to read the output

- **E1** compares SG-DMNMF with SymNMF, DANMF, DMoN-lite, Leiden, fixed alpha, no motif and
  a label-using oracle. `DMRA-stab(a=...)` is the proposed label-free rule.
- **E2** shows clustering quality and perturbation stability against alpha. The selected
  alpha is the stability peak, chosen without labels.
- **E3** covers edge deletion, edge addition and triangle-targeted deletion.
- **E4** removes each model component to measure its contribution.

Synthetic datasets are clearly marked and are demonstration only; the real Dolphin graph
remains unlabeled when no authentic community attribute exists. The written report is in
`overleaf/main.tex` and the slide deck is in `presentation/` in the same repository.